In [69]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from scipy.special import expit
import pandas as pd
import sys
from sklearn.metrics import accuracy_score
import struct
# Load the data
data = pd.read_csv('Leases_WithDates_FixedNA_WithAddresses.csv')

In [ ]:
#data.columns

Index(['year', 'quarter', 'monthsigned', 'market', 'building_name',
       'building_id', 'address', 'region', 'city', 'state', 'zip',
       'internal_submarket', 'internal_class', 'leasedSF', 'company_name',
       'internal_industry', 'transaction_type', 'internal_market_cluster',
       'costarID', 'space_type', 'CBD_suburban', 'RBA', 'available_space',
       'availability_proportion', 'internal_class_rent', 'overall_rent',
       'direct_available_space', 'direct_availability_proportion',
       'direct_internal_class_rent', 'direct_overall_rent',
       'sublet_available_space', 'sublet_availability_proportion',
       'sublet_internal_class_rent', 'sublet_overall_rent', 'leasing',
       'Quarter_Fixed', 'Month_Fixed', 'Address_Fixed'],
      dtype='object')

In [72]:
Month_Fixed_m = {
    '1/1/2018': 1,
    '2/1/2018': 2,
    '3/1/2018': 3,
    '4/1/2018': 4,
    '5/1/2018': 5,
    '6/1/2018': 6,
    '7/1/2018': 7,
    '8/1/2018': 8,
    '9/1/2018': 9,
    '10/1/2018': 10,
    '11/1/2018': 11,
    '12/1/2018': 12,
    '1/1/2019': 13,
    '2/1/2019': 14,
    '3/1/2019': 15,
    '4/1/2019': 16,
    '5/1/2019': 17,
    '6/1/2019': 18,
    '7/1/2019': 19,
    '8/1/2019': 20,
    '9/1/2019': 21,
    '10/1/2019': 22,
    '11/1/2019': 23,
    '12/1/2019': 24,
    '1/1/2020': 25,
    '2/1/2020': 26,
    '3/1/2020': 27,
    '4/1/2020': 28,
    '5/1/2020': 29,
    '6/1/2020': 30,
    '7/1/2020': 31,
    '8/1/2020': 32,
    '9/1/2020': 33,
    '10/1/2020': 34,
    '11/1/2020': 35,
    '12/1/2020': 36,
    '1/1/2021': 37,
    '2/1/2021': 38,
    '3/1/2021': 39,
    '4/1/2021': 40,
    '5/1/2021': 41,
    '6/1/2021': 42,
    '7/1/2021': 43,
    '8/1/2021': 44,
    '9/1/2021': 45,
    '10/1/2021': 46,
    '11/1/2021': 47,
    '12/1/2021': 48,
    '1/1/2022': 49,
    '2/1/2022': 50,
    '3/1/2022': 51,
    '4/1/2022': 52,
    '5/1/2022': 53,
    '6/1/2022': 54,
    '7/1/2022': 55,
    '8/1/2022': 56,
    '9/1/2022': 57,
    '10/1/2022': 58,
    '11/1/2022': 59,
    '12/1/2022': 60,
    '1/1/2023': 61,
    '2/1/2023': 62,
    '3/1/2023': 63,
    '4/1/2023': 64,
    '5/1/2023': 65,
    '6/1/2023': 66,
    '7/1/2023': 67,
    '8/1/2023': 68,
    '9/1/2023': 69,
    '10/1/2023': 70,
    '11/1/2023': 71,
    '12/1/2023': 72,
    '1/1/2024': 73,
    '2/1/2024': 74,
    '3/1/2024': 75,
    '4/1/2024': 76,
    '5/1/2024': 77,
    '6/1/2024': 78,
    '7/1/2024': 79,
    '8/1/2024': 80,
    '9/1/2024': 81,
    '10/1/2024': 82,
    '11/1/2024': 83,
    '12/1/2024': 84
}

Quarter_Fixed_m = {
    '3/1/2018': 1,
    '6/1/2018': 2,
    '9/1/2018': 3,
    '12/1/2018': 4,
    '3/1/2019': 5,
    '6/1/2019': 6,
    '9/1/2019': 7,
    '12/1/2019': 8,
    '3/1/2020': 9,
    '6/1/2020': 10,
    '9/1/2020': 11,
    '12/1/2020': 12,
    '3/1/2021': 13,
    '6/1/2021': 14,
    '9/1/2021': 15,
    '12/1/2021': 16,
    '3/1/2022': 17,
    '6/1/2022': 18,
    '9/1/2022': 19,
    '12/1/2022': 20,
    '3/1/2023': 21,
    '6/1/2023': 22,
    '9/1/2023': 23,
    '12/1/2023': 24,
    '3/1/2024': 25,
    '6/1/2024': 26,
    '9/1/2024': 27,
    '12/1/2024': 28,
}

CBD_suburban_m = {
    'CBD': 0,
    'Suburban': 1
}

space_type_m = {
    'Relet': 1,
    'New': 2,
    'Sublet': 3
}

transaction_type_m = {
    'Expansion': 1,
    'New': 2,
    'Relocation': 3,
    'Renewal': 4,
    'Restructure': 5,
    'Extension': 6,
    'TBD': 7,
    'Renewal and Expansion': 8,
    'Sale - Leaseback': 9
}

internal_class_m = {
    'A': 1,
    'O': 0
}

market_m = {
    'Atlanta': 1,
    'Austin': 2,
    'Baltimore': 3,
    'Boston': 4,
    'Charlotte': 5,
    'Chicago': 6,   
    'Chicago Suburbs': 7,
    'Dallas/Ft Worth': 8,
    'Denver': 9,
    'Detroit': 10,
    'Houston': 11,
    'Los Angeles': 12,
    'Manhattan': 13,
    'Nashville': 14,
    'Northern New Jersey': 15,
    'Northern Virginia': 16,
    'Orange County': 17,
    'Philadelphia': 18,
    'Phoenix': 19,
    'Raleigh/Durham': 20,
    'Salt Lake City': 21,
    'San Diego': 22,
    'San Francisco': 23,
    'Seattle': 24,
    'South Bay/San Jose': 25,
    'South Florida': 26,
    'Southern Maryland': 27,
    'Tampa': 28,
    'Washington D.C.': 29
}

region_m = {
    'South': 1,
    'Northeast': 2,
    'Midwest/Central': 3,
    'West': 4
}

quarter_m = {
    'Q1': 1,
    'Q2': 2,
    'Q3': 3,
    'Q4': 4
}

state_m = {   
    'AL': 1,
    'AK': 2,
    'AZ': 3,
    'AR': 4,
    'CA': 5,
    'CO': 6,
    'CT': 7,
    'DE': 8,
    'FL': 9,
    'GA': 10,
    'HI': 11,
    'ID': 12,
    'IL': 13,
    'IN': 14,
    'IA': 15,
    'KS': 16,
    'KY': 17,
    'LA': 18,
    'ME': 19,
    'MD': 20,
    'MA': 21,
    'MI': 22,
    'MN': 23,
    'MS': 24,
    'MO': 25,
    'MT': 26,
    'NE': 27,
    'NV': 28,
    'NH': 29,
    'NJ': 30,
    'NM': 31,
    'NY': 32,
    'NC': 33,
    'ND': 34,
    'OH': 35,
    'OK': 36,
    'OR': 37,
    'PA': 38,
    'RI': 39,
    'SC': 40,
    'SD': 41,
    'TN': 42,
    'TX': 43,
    'UT': 44,
    'VT': 45,
    'VA': 46,
    'WA': 47,
    'WV': 48,
    'WI': 49,
    'WY': 50,
    'DC': 51,
    'PR': 52
}

In [73]:
data['Month_Fixed'] = data['Month_Fixed'].map(Month_Fixed_m)
data['Quarter_Fixed'] = data['Quarter_Fixed'].map(Quarter_Fixed_m)
data['CBD_suburban'] = data['CBD_suburban'].map(CBD_suburban_m)
data['space_type'] = data['space_type'].map(space_type_m)
data['transaction_type'] = data['transaction_type'].map(transaction_type_m)
data['internal_class'] = data['internal_class'].map(internal_class_m)
data['market'] = data['market'].map(market_m)
data['region'] = data['region'].map(region_m)
data['quarter'] = data['quarter'].map(quarter_m)
data['state'] = data['state'].map(state_m) 

del data['internal_submarket']
del data['internal_market_cluster']
del data['company_name']
del data['building_name']
del data['address']
del data['city']
data['building_id'] = data['building_id'].apply(hash)
data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

In [75]:
from sklearn.preprocessing import StandardScaler

# Your custom list of columns to standardize
cols_to_standardize = ['leasedSF','RBA','available_space','availability_proportion','internal_class_rent', 'overall_rent',
                       'direct_available_space','direct_availability_proportion','direct_internal_class_rent',
                       'direct_overall_rent','sublet_available_space','sublet_availability_proportion',
                       'sublet_internal_class_rent','sublet_overall_rent','leasing']

scaler = StandardScaler()
data[cols_to_standardize] = scaler.fit_transform(data[cols_to_standardize])

In [76]:
industry_to_group = {
    'Financial Services and Insurance': "Professional Services",
    'Construction, Engineering and Architecture': "Hard Industry",
    'Technology, Advertising, Media, and Information': "Professional Services",
    'Manufacturing (except Pharmaceutical, Retail, and Computer Tech)': "Hard Industry",
    'Associations and Non-profit Organizations (except Education and Non-profit Hospitals)': "Public Service",
    'Transportation': "Consumer",
    'Coworking and Executive Suite Companies': "Professional Services",
    'Business, Professional, and Consulting Services (except Financial and Legal) - Including Accounting': "Professional Services",
    'Education': "Public Service",
    'Legal Services': "Professional Services",
    'Real Estate (except coworking providers)': "Professional Services",
    'Healthcare': "Public Service",
    'Personal Services and Recreation': "Consumer",
    'Government': "Public Service",
    'Retail': "Consumer",
    'Energy & Utilities': "Public Service",
    'Pharmaceuticals': "Hard Industry",
    'Agriculture, Forestry, Fishing, Metal & Mineral Mining': "Hard Industry"
}

# Map and filter
data["industry_group"] = data["internal_industry"].map(industry_to_group)

# Define groups
industry_groups = ["Public Service", "Hard Industry", "Consumer", "Professional Services"]

# Create dummy columns
for group in industry_groups:
    data[group] = (data["industry_group"] == group).astype(int)

#remove rows not included in any industry group (unknown or NA)
data = data[data[industry_groups].sum(axis=1) > 0]



In [ ]:
#confirm that coding matches and that unknown and tbd rows are removed
#data["internal_industry"].unique()
#print(data[["industry_group", "internal_industry"]])

array(['Legal Services',
       'Business, Professional, and Consulting Services (except Financial and Legal) - Including Accounting',
       'Real Estate (except coworking providers)',
       'Coworking and Executive Suite Companies',
       'Financial Services and Insurance',
       'Associations and Non-profit Organizations (except Education and Non-profit Hospitals)',
       'Technology, Advertising, Media, and Information', 'Education',
       'Transportation', 'Healthcare', 'Pharmaceuticals',
       'Construction, Engineering and Architecture',
       'Agriculture, Forestry, Fishing, Metal & Mineral Mining',
       'Manufacturing (except Pharmaceutical, Retail, and Computer Tech)',
       'Energy & Utilities', 'Retail', 'Personal Services and Recreation',
       'Government'], dtype=object)

In [82]:
del data['internal_industry']
del data['industry_group']
data

,year,quarter,monthsigned,market,building_id,region,state,zip,internal_class,leasedSF,...,sublet_internal_class_rent,sublet_overall_rent,leasing,Quarter_Fixed,Month_Fixed,Address_Fixed,Public Service,Hard Industry,Consumer,Professional Services
0,2019,1,1.0,1,5095190406666554814,1,10,30303.0,1,-0.535088,...,-0.913764,-1.078577,0.200586,5,13,"100 Peachtree St NW Atlanta, GA 30303",0,0,0,1
1,2019,1,1.0,1,9038003909380173825,1,10,30310.0,0,-0.460572,...,-1.266757,-1.078577,-0.671576,5,13,"2001 Martin Luther King Jr Dr Atlanta, GA 30310",0,0,0,1
2,2019,1,1.0,1,527715664081711645,1,10,30093.0,0,0.129168,...,-1.266757,-1.078577,-0.671576,5,13,"5300 Oakbrook Pky Norcross, GA 30093",0,0,0,1
3,2019,1,1.0,1,2134938337288321847,1,10,30096.0,1,0.734279,...,-0.913764,-1.078577,0.200586,5,13,"3097 Satellite Blvd Duluth, GA 30096",0,0,0,1
4,2019,1,1.0,1,2007848740788302550,1,10,30303.0,1,1.285603,...,-0.913764,-1.078577,0.200586,5,13,"101 Marietta St NW Atlanta, GA 30303",0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12730,2024,4,12.0,23,6578039555880854689,4,5,94107.0,1,0.073272,...,1.203425,1.132349,-0.359953,28,84,"680 Folsom St San Francisco, CA 94107",0,0,0,1
12731,2024,4,12.0,24,6617896997323380800,4,47,98004.0,1,-0.219712,...,-0.092042,-0.213801,-0.760321,28,84,"500 108th Ave NE Bellevue, WA 98004",0,0,0,1
12732,2024,4,12.0,24,2571056907559509874,4,47,98001.0,0,-0.073108,...,-0.370516,-0.213801,-0.906158,28,84,"32001 32nd Ave S Federal Way, WA 98001",0,1,0,0
12733,2024,4,12.0,24,-8424872831536157043,4,47,98004.0,1,-0.527151,...,-0.092042,-0.213801,-0.760321,28,84,"10900 NE 8th St Bellevue, WA 98004",0,0,0,1


In [78]:
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 12174 entries, 0 to 12734
Data columns (total 37 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   year                            12174 non-null  int64  
 1   quarter                         12174 non-null  int64  
 2   monthsigned                     12174 non-null  float64
 3   market                          12174 non-null  int64  
 4   building_id                     12174 non-null  int64  
 5   region                          12174 non-null  int64  
 6   state                           12174 non-null  int64  
 7   zip                             12174 non-null  float64
 8   internal_class                  12174 non-null  int64  
 9   leasedSF                        12174 non-null  float64
 10  internal_industry               12174 non-null  object 
 11  transaction_type                12174 non-null  float64
 12  costarID                        12174

In [79]:
data2 = pd.read_csv('MajorMarketOccupancyData-revised.csv')
print(data2.describe())

FileNotFoundError: [Errno 2] No such file or directory: 'MajorMarketOccupancyData-revised.csv'

In [ ]:
data2.head()

,year,quarter,market,ending_occupancy_proportion,starting_occupancy_proportion,avg_occupancy_proportion
0,2020,Q1,Washington D.C.,0.19,0.98,0.785714
1,2020,Q1,Manhattan,0.08,0.98,0.732857
2,2020,Q1,Chicago,0.14,0.99,0.788571
3,2020,Q1,Houston,0.33,0.99,0.835714
4,2020,Q1,Philadelphia,0.20,0.99,0.817143


In [ ]:
unique = data['year'].unique()
print(unique, " ", len(unique))

[2019 2020 2021 2022 2023 2024]   6
